In an conditional routing agent loop, the model constantly asks, "What should I do next?" 

In a Routing Workflow, the model asks, "Which door should I walk through?" and once it walks through, the process follows a strict, hard-coded path. Imagine you are building a customer service bot:
* You don't want the LLM "experimenting" with how to handle a refund. 
* You want it to recognize a refund request and immediately hand it off to your specific, rock-solid refund script.

In [2]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
llm = ChatOllama(model="llama3.1", temperature=0)

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# 1. Define our two destination paths (Nodes)
def billing_node(state: AgentState):
    """Path A: Handles money problems."""
    response = "Welcome to Billing Support. Let's look at your invoice."
    return {"messages": [("assistant", response)]}

def tech_node(state: AgentState):
    """Path B: Handles technical problems."""
    response = "Welcome to Tech Support. Have you tried turning it off and on?"
    return {"messages": [("assistant", response)]}

# 2. The Classifier (The Conditional Edge Logic)
def classify_and_route(state: AgentState):
    """This isn't a node; it's the intelligence inside the conditional edge."""
    # Grab what the user just said
    user_message = state['messages'][-1].content
    
    # Force the local model to pick a lane
    prompt = f"""
    Is the user asking about money/billing, or technical computer issues? 
    Reply with exactly one word: 'billing' or 'tech'. 
    User Query: {user_message}
    """
    
    # We ask the LLM for its classification
    decision = llm.invoke(prompt).content.lower()
    print(f"🧭 Router LLM decided: {decision.strip()}")
    
    # Defensive routing: if it hallucinates, default to tech
    if "billing" in decision:
        return "billing"
    else:
        return "tech"

# 3. Build the Workflow Graph
workflow = StateGraph(AgentState)

# Add our two department nodes
workflow.add_node("billing_department", billing_node)
workflow.add_node("tech_department", tech_node)

# 4. Draw the Map (The Fork in the Road)
# Notice we route straight from START into the classifier!
workflow.add_conditional_edges(
    START, 
    classify_and_route,
    {
        "billing": "billing_department", # If it returns 'billing', go to Path A
        "tech": "tech_department"        # If it returns 'tech', go to Path B
    }
)

# Both paths lead straight to END. No loops allowed.
workflow.add_edge("billing_department", END)
workflow.add_edge("tech_department", END)

# Compile it
router_app = workflow.compile()

print("Module 3, Step 1 Complete: Deterministic Router compiled!")

/Users/pierson.chu/Downloads/github/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Module 3, Step 1 Complete: Deterministic Router compiled!


In [3]:
# 1. Define our two test cases
test_queries = [
    "Help! I was charged twice for my subscription this month.",
    "My computer screen went totally black and won't turn on."
]

print("Testing the Router...\n")

# 2. Run them through the graph
for query in test_queries:
    print(f"👤 User: '{query}'")
    
    # Create the initial state for each query
    initial_state = {"messages": [("user", query)]}
    
    # Invoke the router app
    final_state = router_app.invoke(initial_state)
    
    # Print the final response
    final_message = final_state["messages"][-1]
    print(f"🤖 Agent: {final_message.content}\n")
    print("-" * 40 + "\n")

print("Module 3 Complete! You now know the difference between Agents and Workflows.")

Testing the Router...

👤 User: 'Help! I was charged twice for my subscription this month.'
🧭 Router LLM decided: billing
🤖 Agent: Welcome to Billing Support. Let's look at your invoice.

----------------------------------------

👤 User: 'My computer screen went totally black and won't turn on.'
🧭 Router LLM decided: tech
🤖 Agent: Welcome to Tech Support. Have you tried turning it off and on?

----------------------------------------

Module 3 Complete! You now know the difference between Agents and Workflows.
